# Week 9: Fair Housing Compliance Screening

Screen proposed listing language for possible preferences, limitations and exclusions. This is a rule-based educational screen, not a legal certification. It does not infer applicant characteristics or publish listings. The project implements error / warning / info levels, a synthetic evaluation, a real-sample scan and a submission-workflow example.

In [1]:
from pathlib import Path
import sys, json
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "scripts/compliance_checker.py").is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import pandas as pd
from IPython.display import display
from scripts.compliance_checker import ComplianceChecker
from scripts.listing_submission import prepare_listing_submission
from scripts.evaluate_week9 import run_evaluation
checker = ComplianceChecker()
print("Python:", sys.executable)
print("Rule library:", checker.version, "| Rules:", len(checker.rules))

Python: /Users/dengyingxin/Desktop/IDX Exchange/NLP intern/.venv/bin/python
Rule library: week9-1.0 | Rules: 25


## 1. Rule library and protected categories

Rules cover race, color, national origin, religion, sex, familial status and disability. The library stores patterns, explanations, suggested editing directions and source URLs. This is not a universal banned-word list. An exclusion such as “no children” is different from white cabinets or a nearby religious facility.

Legal sources: [42 U.S.C. 3604(c)](https://uscode.house.gov/us/usc/t42/s3604/), [24 CFR 100.75](https://www.ecfr.gov/current/title-24/subtitle-B/chapter-I/part-100/subpart-B/section-100.75), and the limited provisions in [42 U.S.C. 3607](https://uscode.house.gov/us/usc/t42/s3607/).

In [2]:
rules = pd.DataFrame(checker.rules)
display(rules.groupby(["category", "severity"]).size().unstack(fill_value=0))
display(rules[["id", "category", "severity", "message"]])

severity,error,info,warning
category,,,
color,1,0,0
disability,3,1,1
familial_status,4,0,2
national_origin,3,0,1
race,3,0,1
religion,2,0,1
sex,1,0,1


,id,category,severity,message
0,familial.no_children,familial_status,error,Excludes households with children.
1,familial.children_not_allowed,familial_status,error,States that households with children are not a...
2,familial.adults_only,familial_status,error,Advertises an adult-only or child-free occupan...
3,familial.unsuitable_children,familial_status,error,Discourages households with children from the ...
4,familial.household_preference,familial_status,warning,Suggests a preferred household profile; review...
5,familial.senior_claim,familial_status,warning,Older-person housing eligibility cannot be ver...
6,disability.no_wheelchairs,disability,error,Excludes wheelchair users.
7,disability.excluded_residents,disability,error,Excludes people because of disability.
8,disability.able_bodied,disability,error,Requires a physical ability profile for reside...
9,disability.animal_policy,disability,warning,Assistance-animal restrictions may require a d...


## 2. Three severity levels and evidence

- error: block this submission for correction.
- warning: require qualified human review; no automatic publication.
- info: explanatory context, not a positive violation detection.

Results include matched text, local context, normalized-text offsets, suggested editing directions and source links. The compatibility key compliant means only that the current screening policy passed. legal_compliance_certified is always false.

In [3]:
examples = [
    "A bright home. No children.",
    "Applicants must be able-bodied.",
    "White tenants only.",
    "Christians only.",
    "Perfect for singles.",
    "55+ community; adults only.",
    "Female roommate wanted.",
    "Wheelchair accessible entrance.",
    "White cabinets and black granite countertops.",
    "A diverse area with ethnic restaurants nearby.",
]
display(pd.DataFrame([{"text": t, "decision": (r := checker.check_listing(t))["decision"], **r["counts"]} for t in examples]))
display(pd.DataFrame(checker.check_listing(examples[0])["findings"]))

,text,decision,error,warning,info
0,A bright home. No children.,block,1,0,0
1,Applicants must be able-bodied.,block,1,0,0
2,White tenants only.,block,1,0,0
3,Christians only.,block,1,0,0
4,Perfect for singles.,review,0,1,0
5,55+ community; adults only.,review,0,2,0
6,Female roommate wanted.,review,0,1,0
7,Wheelchair accessible entrance.,pass,0,0,1
8,White cabinets and black granite countertops.,pass,0,0,0
9,A diverse area with ethnic restaurants nearby.,pass,0,0,0


,rule_id,category,severity,pattern,matched_text,span,context,context_reason,message,suggestion,sources
0,familial.no_children,familial_status,error,\b(?:no|without)\s+(?:young\s+)?(?:children|ki...,No children,"[15, 26]",A bright home. No children.,None,Excludes households with children.,"Describe the property, not which households ma...","[https://uscode.house.gov/us/usc/t42/s3604/, h..."


## 3. Context and false-positive controls

Normalization handles case, whitespace, HTML entities/tags, dash variants and Unicode width variants. Explicitly rejected phrases and clear historical wording with an inclusive replacement retain informational matches. An owner quoting a current exclusion still produces a risk finding. An inclusive slogan cannot erase an exclusion elsewhere.

Senior housing, claimed religious-organization housing and shared-living phrases are routed to review; a keyword never proves a legal exception. No automated human override is implemented.

In [4]:
context_examples = [
    'Do not use the phrase "no children".',
    'Previously "white tenants only", now everyone is welcome.',
    "All are welcome. No children.",
    "The owner says 'no children'.",
    "No wheelchair ramp is installed.",
    "Christian tenants only. Owned by a religious organization.",
]
display(pd.DataFrame([{"text": t, "decision": (r := checker.check_listing(t))["decision"], "context_reasons": [f["context_reason"] for f in r["findings"]], **r["counts"]} for t in context_examples]))

,text,decision,context_reasons,error,warning,info
0,"Do not use the phrase ""no children"".",pass,[explicit_rejection_or_historical_quote],0,0,1
1,"Previously ""white tenants only"", now everyone ...",pass,[explicit_rejection_or_historical_quote],0,0,1
2,All are welcome. No children.,block,[None],1,0,0
3,The owner says 'no children'.,block,[None],1,0,0
4,No wheelchair ramp is installed.,pass,[],0,0,0
5,Christian tenants only. Owned by a religious o...,review,[religious_organization_claim_unverified],0,1,0


## 4. Evaluation protocol and actual results

The development set has 68 examples and the test set 103.   
The test set includes 41 known violations, 17 cases requiring review, and 45 clean cases.  

Labels are assistant-authored synthetic judgments, not independently human-verified legal labels. The texts are disjoint but phrase families overlap; the fixtures were used during development. They are regression tests, not a blind deployment benchmark.

Case-level screening counts error or warning as positive; info alone is negative. Both known_violation and review labels require a flag. The known-violation recall target is reported separately, along with error-only recall, per-category recall, precision, F1 and decision accuracy. All mismatches are retained. A perfect synthetic score would not establish real-world perfection.

In [5]:
report = run_evaluation()
metric_rows = []
for split, r in report["splits"].items():
    metric_rows.append({"split": split, "cases": r["count"], **r["confusion"], "precision": r["screening_precision"], "screening_recall": r["screening_recall"], "known_violation_recall": r["known_violation_recall"], "known_violation_error_recall": r["known_violation_error_recall"], "decision_accuracy": r["decision_accuracy"]})
display(pd.DataFrame(metric_rows))
print("Test targets:", report["splits"]["test"]["targets_met"])
print("Status:", report["evaluation_status"])
display(pd.DataFrame(report["splits"]["test"]["per_category_known_violation_recall"]).T)

,split,cases,tp,fp,fn,tn,precision,screening_recall,known_violation_recall,known_violation_error_recall,decision_accuracy
0,development,68,38,0,0,30,1.0,1.0,1.0,1.0,1.0
1,test,103,58,0,0,45,1.0,1.0,1.0,1.0,1.0


Test targets: {'known_violation_recall_100': True, 'precision_over_80': True}
Status: synthetic_regression_only_not_human_verified


,support,detected,recall
familial_status,6.0,6.0,1.0
disability,6.0,6.0,1.0
race,6.0,6.0,1.0
color,5.0,5.0,1.0
national_origin,6.0,6.0,1.0
religion,6.0,6.0,1.0
sex,6.0,6.0,1.0


## 5. Screening real listing remarks and Week 8 summaries

The evaluator also screens the Week 1 sample and any saved Week 8 summaries. These records are unlabeled for this task: flag counts are not proven violation counts, and no precision/recall is calculated for them. Older-person housing and other review flags may be legitimate descriptions requiring verification.

The complete findings are saved locally in data/processed/week9_sample_screening.jsonl. They do not modify listing descriptions, change Week 8 scores, or publish anything.

In [6]:
print(json.dumps(report.get("real_sample_scan", {"status": "no local sample available"}), indent=2))
scan_path = ROOT / "data/processed/week9_sample_screening.jsonl"
if scan_path.exists():
    scan = [json.loads(line) for line in scan_path.read_text().splitlines()]
    flags = [{"listing_id": r["listing_id"], "field": field, "decision": result["decision"], "finding": f["matched_text"], "severity": f["severity"], "reason": f["message"]} for r in scan for field, result in r["checks"].items() for f in result["findings"] if f["severity"] != "info"]
    display(pd.DataFrame(flags).head(12))

{
  "listing_count": 1000,
  "listings_flagged_for_correction_or_review": 19,
  "decisions_by_field": {
    "remarks": {
      "pass": 981,
      "review": 19
    },
    "summary": {
      "pass": 987,
      "review": 13
    }
  },
  "ground_truth_available": false,
  "interpretation": "Counts of screening flags, not counts of proven violations; no precision or recall is calculated on unlabeled real listings."
}


,listing_id,field,decision,finding,severity,reason
0,1178695849,remarks,review,55+,warning,Older-person housing eligibility cannot be ver...
1,1178695849,summary,review,55+,warning,Older-person housing eligibility cannot be ver...
2,1158622225,remarks,review,55+,warning,Older-person housing eligibility cannot be ver...
3,1158622225,remarks,review,55+,warning,Older-person housing eligibility cannot be ver...
4,1173655011,remarks,review,senior community,warning,Older-person housing eligibility cannot be ver...
5,1173655011,remarks,review,55+,warning,Older-person housing eligibility cannot be ver...
6,1173655011,summary,review,senior community,warning,Older-person housing eligibility cannot be ver...
7,1176312992,remarks,review,55+,warning,Older-person housing eligibility cannot be ver...
8,1176312992,summary,review,55+,warning,Older-person housing eligibility cannot be ver...
9,1176990471,remarks,review,SENIOR COMMUNITY,warning,Older-person housing eligibility cannot be ver...


## 6. Listing submission workflow with Week 8

prepare_listing_submission checks original remarks before generating a summary. If the original description is blocked or requires review, a neutral summary cannot make it pass. A supplied or generated summary is also checked. The result is blocked, pending_review or ready_for_submission. published remains false: this is a local simulation, not a publishing service.

In [7]:
submission_examples = [
    {"L_ListingID": "demo-block", "remarks": "A bright home. No children.", "summary": "A bright home."},
    {"L_ListingID": "demo-review", "remarks": "55+ community; adults only."},
    {"L_ListingID": "demo-ready", "L_City": "Irvine", "beds": 3, "baths": 2, "price": 950000, "remarks": "This Irvine home has 3 bedrooms and 2 bathrooms. A private pool is included."},
]
submissions = [prepare_listing_submission(r, checker=checker) for r in submission_examples]
display(pd.DataFrame([{k: r[k] for k in ("listing_id", "status", "can_submit", "published", "summary", "message")} for r in submissions]))

,listing_id,status,can_submit,published,summary,message
0,demo-block,blocked,False,False,A bright home.,Correct the flagged language before resubmitting.
1,demo-review,pending_review,False,False,None,Qualified human review is required before subm...
2,demo-ready,ready_for_submission,True,False,This Irvine home has 3 bedrooms and 2 bathroom...,Language screening passed; continue the remain...


## 7. Try another description

In [8]:
import ipywidgets as widgets
text_input = widgets.Textarea(value="A bright 3-bedroom home. No children.", description="Description:", rows=4, layout=widgets.Layout(width="100%"), style={"description_width": "95px"})
check_button = widgets.Button(description="Check language", button_style="primary")
check_output = widgets.Output()
def check_description(_=None):
    with check_output:
        check_output.clear_output(wait=True)
        result = checker.check_listing(text_input.value)
        print("Decision:", result["decision"], "|", result["counts"])
        print("Legal compliance certified:", result["legal_compliance_certified"])
        if result["findings"]:
            display(pd.DataFrame(result["findings"])[["category", "severity", "matched_text", "message", "suggestion"]])
check_button.on_click(check_description)
display(widgets.VBox([text_input, check_button, check_output]))
check_description()